# Korean Name Sign Language (Challenge 3) — Client

What you fill in: `load_model()` and `run_inference(model, data)`. Everything else (registration, round tracking, submission) is provided.

**Prediction format**

- `prediction`: composed Korean name as a string (e.g. `"박상민"`),
  or a dict like `{"current_char":"민","composed_text":"박상민"}`.
- `visualizations`: format-free (PIL Image recommended for the dashboard)


**Workflow**
1. TA gives you `SERVER_URL` and your `TEAM_SECRET` (one per team).
2. Run the registration cell — the client exchanges the secret for a session token automatically.
3. When the TA calls `start_round(r)`, you call `process_round()`.


In [ ]:
# 1. Dependencies
!pip install -q requests pillow


In [ ]:
!apt-get install -qq fonts-nanum

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import requests, base64, io, os, glob
from PIL import Image

font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fm.fontManager.addfont(font_path)


plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 2. Configuration

# [NOTICE] The SERVER_URL and TEAM_SECRET will be provided by the TA on the demo day.
# The current dashboard link will expire soon.
SERVER_URL  = 'https://writes-tutorials-workshop-challenged.trycloudflare.com' # from TA
TEAM_SECRET = 'token_teamN_xxxxxxxx' # from TA (your team only)

KOR_CONSONANTS = ['ㄱ','ㄴ','ㄷ','ㄹ','ㅁ','ㅂ','ㅅ','ㅇ','ㅈ','ㅊ','ㅋ','ㅌ','ㅍ','ㅎ']
KOR_VOWELS     = ['ㅏ','ㅐ','ㅑ','ㅒ','ㅓ','ㅔ','ㅕ','ㅖ','ㅗ','ㅛ','ㅜ','ㅠ','ㅡ','ㅢ','ㅣ','ㅘ','ㅝ']
ALL_JAMO = KOR_CONSONANTS + KOR_VOWELS

# For practice: You can upload your own collected image/sequence data
# to the following path (eval/latest/) to test your model freely before the demo.
# On the demo day, this path might be updated to point to the official evaluation set.
TEST_DATA_DIR = '/content/drive/MyDrive/dl_challenge1/eval/' # (example)


In [ ]:
# 3. Register with the server (run once per session)

# Module-level token storage; submit_to_server() picks it up automatically.
_session_token = None
_team_id = None

def register():
    global _session_token, _team_id
    r = requests.post(f'{SERVER_URL}/register', json={'secret': TEAM_SECRET}, timeout=10)
    if r.status_code == 401:
        raise RuntimeError('TEAM_SECRET is invalid. Ask the TA for the correct secret.')
    r.raise_for_status()
    data = r.json()
    _session_token = data['token']
    _team_id = data['team']
    print(f'Registered as {_team_id} (token reused: {data["reused"]})')
    return _team_id

register()


Registered as team1 (token reused: False)


'team1'

In [ ]:
# 5. (Optional) Mount Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 6. Load your model
def load_model():
    # TODO: jamo classifier
    return None

model = load_model()


In [ ]:
def load_eval_data():
    return sorted(glob.glob(TEST_DATA_DIR + '*.png') + glob.glob(TEST_DATA_DIR + '*.jpg'))

def classify_jamo(image_path):
    # TODO: Model Inference Logic
    return 'ㄱ' # Output Example

def run_inference():
    image_paths = load_eval_data()

    current_char = "ㄴ"
    composed_text = "박상민"

    fig, (ax_text, ax_img) = plt.subplots(nrows=2, ncols=1, figsize=(6, 6), gridspec_kw={'height_ratios': [1, 3]})
    fig.patch.set_facecolor('#1a1d24')

    ax_text.set_facecolor('#1a1d24')
    ax_text.axis('off')
    ax_text.text(0.2, 0.7, f"Current Char: '{current_char}'",
                 ha='center', va='center', fontsize=24, color='#fbbc04', fontweight='bold')
    ax_text.text(0.65, 0.7, f"Full Name: {composed_text}",
                 ha='center', va='center', fontsize=24, color='#7ee399', fontweight='bold')


    ax_img.set_facecolor('#1a1d24')
    ax_img.axis('off')
    if image_paths:
        latest_image_path = image_paths[-1]
        img = Image.open(latest_image_path)
        ax_img.imshow(img)
    else:
        ax_img.text(0.5, 0.5, "None", ha='center', va='center', color='#9aa0a6')

    plt.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.close(fig)
    buf.seek(0)

    # Server submission format (Dict required for Challenge 3)
    prediction_data = {
        "current_char": current_char,
        "composed_text": composed_text
    }

    return prediction_data, Image.open(buf)

In [ ]:
# Submission helper (token-based, viz auto-encoded)

def _encode_viz(image):
    """Accepts PIL Image / numpy ndarray / base64 str / None."""
    if image is None:
        return None
    if isinstance(image, str):
        return image
    try:
        from PIL import Image
        if not isinstance(image, Image.Image):
            import numpy as np
            arr = image
            if arr.dtype != 'uint8':
                arr = (arr * 255).clip(0, 255).astype('uint8')
            image = Image.fromarray(arr)
        if max(image.size) > 640:
            ratio = 640 / max(image.size)
            image = image.resize((int(image.size[0]*ratio), int(image.size[1]*ratio)))
        buf = io.BytesIO()
        image.save(buf, format='PNG', optimize=True)
        return base64.b64encode(buf.getvalue()).decode('ascii')
    except Exception as e:
        print(f'[viz encode failed] {e}')
        return None

def submit_to_server(prediction, viz_image=None, timeout=10):
    if _session_token is None:
        raise RuntimeError('Not registered. Call register() first.')
    st = requests.get(f'{SERVER_URL}/api/state', timeout=5).json()
    round_id = st['current_round']
    if st['status'] != 'active':
        raise RuntimeError(f'Round {round_id} is {st["status"]} — cannot submit.')
    payload = {
        'round_id': round_id,
        'prediction': prediction,
        'viz_b64': _encode_viz(viz_image),
    }
    r = requests.post(
        f'{SERVER_URL}/submit',
        headers={'X-Team-Token': _session_token},
        json=payload, timeout=timeout,
    )
    if r.status_code == 423:
        raise RuntimeError(f'Round {round_id} closed (timer expired or TA closed it).')
    r.raise_for_status()
    return r.json()


In [ ]:
# Process one round
def process_round():
    """Run this function when the TA starts a new round."""
    pred, viz = run_inference()

    resp = submit_to_server(pred, viz_image=viz)
    print(f'[submitted] pred="{pred}" resp={resp}')
    return resp

In [ ]:
process_round()


[submitted] pred="{'current_char': 'ㄴ', 'composed_text': '박상민'}" resp={'ok': True, 'round': 1, 'team': 'team1'}


{'ok': True, 'round': 1, 'team': 'team1'}